# Train VGG16 on Colab - Simple Version
Notebook đơn giản - chỉ để train model với GPU

**Yêu cầu**: Data Mining đã chạy trên máy local và push lên Git

## 1. Clone Repository

In [ ]:
# Thay đổi thông tin repo
REPO_URL = "https://github.com/OWNER/REPO_NAME.git"
BRANCH_NAME = "vinhkhoa"
REPO_NAME = "Project_DBM"

!git clone {REPO_URL}
%cd {REPO_NAME}
!git checkout {BRANCH_NAME}

## 2. Check GPU (BẮT BUỘC)

In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    print("\n⚠️ WARNING: GPU not available!")
    print("Go to: Runtime > Change runtime type > Select GPU (T4)")

## 3. Install Dependencies

In [ ]:
!pip install -q -r requirements.txt
print("✓ Dependencies installed")

## 4. Upload Dataset

Chọn 1 trong 2 options:

### Option A: Mount Google Drive (Recommended)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Copy dataset từ Drive
# Dataset ở: /content/drive/MyDrive/DBM/
!cp -r /content/drive/MyDrive/DBM/train ./train
!cp /content/drive/MyDrive/DBM/labels.csv ./labels.csv

# (Optional) Copy test nếu cần
# !cp -r /content/drive/MyDrive/DBM/test ./test

print("✓ Dataset copied from Drive")

### Option B: Upload ZIP file

In [ ]:
# from google.colab import files
# uploaded = files.upload()  # Upload dataset.zip
# !unzip -q dataset.zip
# print("✓ Dataset extracted")

## 5. Verify Dataset

In [ ]:
import os
import pandas as pd

# Check files
if os.path.exists('labels.csv'):
    df = pd.read_csv('labels.csv')
    print(f"✓ labels.csv: {len(df)} samples, {df['breed'].nunique()} breeds")
else:
    print("❌ labels.csv NOT FOUND!")

if os.path.exists('train'):
    num_images = len(os.listdir('train'))
    print(f"✓ train/: {num_images} images")
else:
    print("❌ train/ directory NOT FOUND!")

# Check data mining results (optional)
if os.path.exists('data_mining_results/mining_report.json'):
    print("✓ Data mining results found (from local)")
else:
    print("⚠️ Data mining results not found (optional)")

## 6. 🚀 TRAIN VGG16

**Đây là bước chính - chỉ cần chạy lệnh này!**

In [ ]:
!python train_vgg.py

## 7. View Results

In [ ]:
import json

# Load metrics
with open('training_models/vgg_test_metrics.json', 'r') as f:
    metrics = json.load(f)

print("="*60)
print("  VGG16 RESULTS")
print("="*60)
print(f"Accuracy:  {metrics['test_accuracy']:.4f} ({metrics['test_accuracy']*100:.2f}%)")
print(f"Precision: {metrics['test_precision']:.4f}")
print(f"Recall:    {metrics['test_recall']:.4f}")
print(f"F1-Score:  {metrics['test_f1']:.4f}")
print("="*60)

## 8. Download Results

In [ ]:
from google.colab import files

# Download model và metrics
files.download('training_models/best_vgg.pth')
files.download('training_models/vgg_test_metrics.json')
files.download('training_models/vgg_training_history.json')

print("✓ Files downloaded!")

## 9. Backup to Drive

In [ ]:
# Backup toàn bộ kết quả vào Drive
# Lưu vào folder DBM để dễ quản lý
!mkdir -p /content/drive/MyDrive/DBM/results
!cp -r training_models /content/drive/MyDrive/DBM/results/
!cp -r tensorboard_vgg /content/drive/MyDrive/DBM/results/

print("✓ Backup completed!")
print("Location: /content/drive/MyDrive/DBM/results/")

## 10. (Optional) TensorBoard

In [ ]:
%load_ext tensorboard
%tensorboard --logdir tensorboard_vgg

---

## Next Steps

### Train more models:
```python
!python train_resnet.py
!python train_efficientnet.py
```

### Compare models:
```python
!python compare_models.py
```

### Resume training (if interrupted):
```python
# Just run again - it will load last checkpoint
!python train_vgg.py
```